# Praktikum 1: Menyiapkan Repositori dan Mengenal Data
**Catatan Eksplorasi yang Berorientasi Keputusan**
Dataset: Kredit Sumatera (Seed 42)

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

if not Path('data/raw/kredit_sumatera.csv').exists() and Path('../data/raw/kredit_sumatera.csv').exists():
    os.chdir('..')

df = pd.read_csv('data/raw/kredit_sumatera.csv')
print(f'Total baris : {len(df):,}')
print(f'Periode     : {df.periode.min()} s.d. {df.periode.max()}')

Total baris : 32,400
Periode     : 2022-01 s.d. 2024-12


## Langkah 4: Eksplorasi yang Berorientasi Keputusan

In [2]:
# Pertanyaan 1: Seberapa tidak seimbang targetnya?
print('Tingkat gagal bayar:', f'{df.gagal_bayar.mean():.2%}')
print(df.gagal_bayar.value_counts(normalize=True).round(4))

Tingkat gagal bayar: 8.30%
gagal_bayar
0    0.917
1    0.083
Name: proportion, dtype: float64


In [3]:
# Pertanyaan 2: Apakah populasi berubah sepanjang waktu?
print(pd.crosstab(df.periode.str[:4], df.segmen, normalize='index').round(3))

segmen   nelayan  pedagang_musiman  pegawai_tetap  petani  wiraswasta
periode                                                              
2022       0.067             0.139          0.421    0.13       0.242
2023       0.067             0.142          0.419    0.13       0.241
2024       0.099             0.171          0.339    0.16       0.231


In [4]:
# Pertanyaan 3: Kolom mana yang punya nilai hilang, dan mengapa?
hilang = df.isna().mean().sort_values(ascending=False)
print('Persentase nilai hilang keseluruhan:')
print(hilang[hilang > 0].round(4))

print('\nProporsi nilai hilang rata_saldo_90h per segmen:')
print(df.groupby('segmen')['rata_saldo_90h'].apply(lambda s: s.isna().mean()).round(3))

print('\nProporsi nilai hilang lama_usaha_tahun per segmen:')
print(df.groupby('segmen')['lama_usaha_tahun'].apply(lambda s: s.isna().mean()).round(3))

Persentase nilai hilang keseluruhan:
status_restrukturisasi    0.9479
lama_usaha_tahun          0.0294
rata_saldo_90h            0.0231
dtype: float64

Proporsi nilai hilang rata_saldo_90h per segmen:
segmen
nelayan             0.108
pedagang_musiman    0.000
pegawai_tetap       0.000
petani              0.105
wiraswasta          0.000
Name: rata_saldo_90h, dtype: float64

Proporsi nilai hilang lama_usaha_tahun per segmen:
segmen
nelayan             0.025
pedagang_musiman    0.030
pegawai_tetap       0.030
petani              0.028
wiraswasta          0.030
Name: lama_usaha_tahun, dtype: float64


In [5]:
# Pertanyaan 4: Adakah kolom yang mencurigakan?
print('Proporsi status_restrukturisasi terisi berdasarkan gagal_bayar:')
print(df.groupby('gagal_bayar')['status_restrukturisasi'].apply(lambda s: s.notna().mean()))
print('\nTabel silang gagal_bayar vs status_restrukturisasi:')
print(pd.crosstab(df.gagal_bayar, df.status_restrukturisasi.fillna('KOSONG'), margins=True))

Proporsi status_restrukturisasi terisi berdasarkan gagal_bayar:
gagal_bayar
0    0.000000
1    0.627509
Name: status_restrukturisasi, dtype: float64

Tabel silang gagal_bayar vs status_restrukturisasi:
status_restrukturisasi  KOSONG    ya    All
gagal_bayar                                
0                        29710     0  29710
1                         1002  1688   2690
All                      30712  1688  32400


## Langkah 5: Model Garis Dasar (Baseline) Out-of-Time Split

In [6]:
df['rasio_angsuran'] = df.angsuran_bulanan / df.penghasilan_bulanan
num = ['usia', 'tanggungan', 'lama_usaha_tahun', 'penghasilan_bulanan',
       'ragam_penghasilan', 'plafon_diajukan', 'tenor_bulan', 'bunga_tahunan',
       'rasio_angsuran', 'rata_saldo_90h', 'jumlah_tunggakan_12b']
X = df[num].join(pd.get_dummies(df[['segmen', 'wilayah', 'kanal']]))
y = df.gagal_bayar

# Pembagian MENURUT WAKTU (Out-of-Time)
latih = df.periode <= '2023-12'
uji = df.periode > '2024-06'

m = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06, max_iter=250, random_state=42)
m.fit(X[latih], y[latih])
skor = m.predict_proba(X[uji])[:, 1]
print('AUC out-of-time:', round(roc_auc_score(y[uji], skor), 4))

AUC out-of-time: 0.7744


## Langkah 6: Membandingkan dengan Pembagian Acak (Random Stratified Split)

In [7]:
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m2 = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06, max_iter=250, random_state=42).fit(Xa, ya)
print('AUC pembagian acak:', round(roc_auc_score(yb, m2.predict_proba(Xb)[:, 1]), 4))

AUC pembagian acak: 0.7631


## Uji Efek Kebocoran Target (status_restrukturisasi)

In [8]:
# Eksperimen peragaan kebocoran fitur status_restrukturisasi
X_bocor = X.copy()
X_bocor['bocor_restrukturisasi'] = df['status_restrukturisasi'].notna().astype(float)
m_bocor = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.06, max_iter=250, random_state=42)
m_bocor.fit(X_bocor[latih], y[latih])
skor_bocor = m_bocor.predict_proba(X_bocor[uji])[:, 1]
print('AUC out-of-time DENGAN fitur bocor:', round(roc_auc_score(y[uji], skor_bocor), 4))

AUC out-of-time DENGAN fitur bocor: 0.9063
